# 🩺 Diabetes Dataset — Phân Tích & Tiền Xử Lý Dữ Liệu cho XAI

**Dataset:** Pima Indians Diabetes Database  
**Task:** Binary Classification — Explainable AI (XAI)  
**Mục tiêu:** Phân tích, làm sạch và chuẩn bị dữ liệu cho pipeline XAI


## 0. Cài Đặt Thư Viện & Import

In [ ]:
# Cài đặt thư viện cần thiết (chạy 1 lần)
# !pip install pandas numpy matplotlib seaborn scikit-learn missingno

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer

# Cấu hình hiển thị
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('Set2')

print("✅ Import thành công!")


## 1. Load Dữ Liệu

In [ ]:
# Load dataset
df = pd.read_csv('diabetes.csv')

print(f"📐 Kích thước dataset: {df.shape[0]} mẫu × {df.shape[1]} đặc trưng")
print(f"\n📋 Các cột: {df.columns.tolist()}")
print("\n--- 5 Dòng Đầu ---")
df.head()


## 2. Mô Tả Đặc Trưng (Feature Definition)

| Feature | Đơn vị | Mô tả |
|---------|--------|-------|
| `Pregnancies` | lần | Số lần mang thai |
| `Glucose` | mg/dL | Nồng độ glucose huyết tương (oral glucose tolerance test) |
| `BloodPressure` | mmHg | Huyết áp tâm trương |
| `SkinThickness` | mm | Độ dày nếp gấp da cơ tam đầu |
| `Insulin` | mu U/ml | Nồng độ insulin huyết thanh 2 giờ |
| `BMI` | kg/m² | Chỉ số khối cơ thể |
| `DiabetesPedigreeFunction` | — | Hàm phả hệ tiểu đường (yếu tố di truyền) |
| `Age` | năm | Tuổi |
| **`Outcome`** | 0/1 | **Target**: 1 = tiểu đường, 0 = không |


## 3. Phân Tích Khám Phá Dữ Liệu (EDA)

In [ ]:
# --- 3.1 Thống kê mô tả ---
print("📊 Thống Kê Mô Tả:")
df.describe().round(3)


In [ ]:
# --- 3.2 Kiểm tra kiểu dữ liệu & null ---
print("📋 Kiểu dữ liệu:")
print(df.dtypes)
print(f"\n❓ Giá trị null thực sự: {df.isnull().sum().sum()} (nhưng có giá trị 0 mã hóa missing!)")


In [ ]:
# --- 3.3 Phân bố nhãn (Class Distribution) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
outcome_counts = df['Outcome'].value_counts()
bars = axes[0].bar(['Không tiểu đường (0)', 'Tiểu đường (1)'], 
                    outcome_counts.values, 
                    color=['#2ecc71', '#e74c3c'], alpha=0.85, edgecolor='white')
for bar, val in zip(bars, outcome_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
                 f'{val}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=12)
axes[0].set_title('Phân Bố Nhãn (Outcome)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Số lượng mẫu')
axes[0].set_ylim(0, 600)

# Pie chart
axes[1].pie(outcome_counts.values, labels=['Không (0)', 'Có (1)'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Tỷ Lệ Mắc Tiểu Đường', fontsize=13, fontweight='bold')

plt.suptitle('Class Distribution — Diabetes Dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n⚠️  Mất cân bằng nhãn: {outcome_counts[0]}/{outcome_counts[1]} = {outcome_counts[0]/outcome_counts[1]:.2f}:1")


In [ ]:
# --- 3.4 Phân phối từng đặc trưng ---
features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
            'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    ax = axes[i]
    # Histogram by class
    df[df['Outcome']==0][col].hist(ax=ax, alpha=0.6, color='#2ecc71', 
                                    bins=25, label='Outcome=0', density=True)
    df[df['Outcome']==1][col].hist(ax=ax, alpha=0.6, color='#e74c3c', 
                                    bins=25, label='Outcome=1', density=True)
    ax.set_title(col, fontweight='bold', fontsize=11)
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Phân Phối Đặc Trưng theo Nhãn (Outcome)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- 3.5 Ma trận tương quan (Correlation Matrix) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full correlation
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=axes[0], linewidths=0.5,
            cbar_kws={'label': 'Correlation'})
axes[0].set_title('Ma Trận Tương Quan (Pearson)', fontweight='bold')

# Correlation with target
target_corr = df.corr()['Outcome'].drop('Outcome').sort_values()
colors = ['#e74c3c' if v > 0 else '#3498db' for v in target_corr.values]
axes[1].barh(target_corr.index, target_corr.values, color=colors, alpha=0.8)
axes[1].axvline(x=0, color='black', linewidth=0.8)
axes[1].set_title('Tương Quan với Outcome (Target)', fontweight='bold')
axes[1].set_xlabel('Pearson Correlation')
for i, (idx, val) in enumerate(target_corr.items()):
    axes[1].text(val + 0.005 * np.sign(val), i, f'{val:.3f}', 
                 va='center', fontsize=10)

plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🔑 Top 3 đặc trưng tương quan cao nhất với Outcome:")
print(df.corr()['Outcome'].drop('Outcome').abs().sort_values(ascending=False).head(3))


## 4. Phát Hiện & Xử Lý Giá Trị Zero Không Hợp Lệ

In [ ]:
# --- 4.1 Phân tích zero values ---
# Các cột có zero không hợp lệ về mặt y học
ZERO_INVALID_COLS = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print("🔍 Phân Tích Giá Trị Zero Không Hợp Lệ:")
print("=" * 55)
zero_report = []
for col in ZERO_INVALID_COLS:
    n_zeros = (df[col] == 0).sum()
    pct = n_zeros / len(df) * 100
    severity = "🔴 NGHIÊM TRỌNG" if pct > 20 else ("🟡 TRUNG BÌNH" if pct > 5 else "🟢 THẤP")
    zero_report.append({'Feature': col, 'Số Zero': n_zeros, 'Tỷ Lệ (%)': f'{pct:.1f}%', 'Mức Độ': severity})
    print(f"{col:25s}: {n_zeros:3d} zeros ({pct:5.1f}%) — {severity}")

print("\n✅ Pregnancies = 0: HỢP LỆ (chưa từng mang thai)")


In [ ]:
# --- 4.2 Visualize zero values ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart - tỷ lệ zero
zero_pcts = [(df[col] == 0).sum() / len(df) * 100 for col in ZERO_INVALID_COLS]
colors_zero = ['#e74c3c' if p > 20 else ('#f39c12' if p > 5 else '#27ae60') for p in zero_pcts]
bars = axes[0].bar(ZERO_INVALID_COLS, zero_pcts, color=colors_zero, alpha=0.85, edgecolor='white')
axes[0].axhline(y=20, color='red', linestyle='--', alpha=0.7, label='Ngưỡng nghiêm trọng (20%)')
axes[0].axhline(y=5, color='orange', linestyle='--', alpha=0.7, label='Ngưỡng trung bình (5%)')
for bar, val in zip(bars, zero_pcts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Tỷ Lệ Giá Trị Zero theo Feature', fontweight='bold')
axes[0].set_ylabel('Tỷ lệ (%)')
axes[0].legend()

# Heatmap missing pattern (zero = missing)
df_missing = df[ZERO_INVALID_COLS].copy()
df_missing = df_missing.replace(0, np.nan)
missing_by_outcome = []
for outcome in [0, 1]:
    subset = df_missing[df['Outcome'] == outcome]
    missing_pct = subset.isnull().mean() * 100
    missing_by_outcome.append(missing_pct)
missing_df = pd.DataFrame(missing_by_outcome, index=['Outcome=0', 'Outcome=1'])
sns.heatmap(missing_df, annot=True, fmt='.1f', cmap='Reds', ax=axes[1],
            linewidths=0.5, cbar_kws={'label': '% Missing'})
axes[1].set_title('Tỷ Lệ Missing (%) theo Class', fontweight='bold')

plt.tight_layout()
plt.savefig('missing_values_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Làm Sạch Dữ Liệu (Data Cleaning)

In [ ]:
# --- 5.1 Thay thế zero → NaN ---
df_clean = df.copy()

for col in ZERO_INVALID_COLS:
    df_clean[col] = df_clean[col].replace(0, np.nan)

print("✅ Đã thay thế zero → NaN cho các cột y tế")
print(f"   Tổng số NaN sau khi thay thế: {df_clean.isnull().sum().sum()}")
print("\nSố NaN theo cột:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])


In [ ]:
# --- 5.2 Imputation: Median theo Outcome (cho cột ít missing) ---
# Glucose, BloodPressure, BMI: median imputation theo nhóm Outcome
LOW_MISSING_COLS = ['Glucose', 'BloodPressure', 'BMI']

for col in LOW_MISSING_COLS:
    medians = df_clean.groupby('Outcome')[col].median()
    for outcome_val in [0, 1]:
        mask = (df_clean['Outcome'] == outcome_val) & (df_clean[col].isnull())
        df_clean.loc[mask, col] = medians[outcome_val]
    print(f"✅ {col}: imputed bằng median theo Outcome "
          f"(Outcome=0: {medians[0]:.1f}, Outcome=1: {medians[1]:.1f})")


In [ ]:
# --- 5.3 KNN Imputation (cho SkinThickness & Insulin - nhiều missing) ---
# Sử dụng KNN để học từ các đặc trưng tương quan
HIGH_MISSING_COLS = ['SkinThickness', 'Insulin']

print("⚙️  Đang chạy KNN Imputer (k=5)... ", end='')
knn_imputer = KNNImputer(n_neighbors=5, weights='distance')

# Fit & transform trên toàn bộ dataset (trừ Outcome)
feature_cols = [c for c in df_clean.columns if c != 'Outcome']
df_clean_values = knn_imputer.fit_transform(df_clean[feature_cols])
df_knn = pd.DataFrame(df_clean_values, columns=feature_cols)
df_clean[feature_cols] = df_knn
df_clean['Outcome'] = df['Outcome'].values  # Đảm bảo Outcome là int

print("✅ Hoàn thành!")
print(f"\n✅ Tổng NaN sau imputation: {df_clean.isnull().sum().sum()}")


## 6. Phát Hiện & Xử Lý Outlier

In [ ]:
# --- 6.1 Phát hiện outlier bằng IQR ---
def detect_outliers_iqr(data, col, multiplier=1.5):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    outliers = ((data[col] < lower) | (data[col] > upper)).sum()
    return lower, upper, outliers

feature_cols_no_outcome = [c for c in df_clean.columns if c != 'Outcome']

print("🔍 Phát Hiện Outlier (IQR × 1.5):")
print(f"{'Feature':<28} {'Lower':>8} {'Upper':>8} {'Outliers':>10} {'Tỷ lệ':>8}")
print("-" * 65)
outlier_summary = {}
for col in feature_cols_no_outcome:
    low, up, n_out = detect_outliers_iqr(df_clean, col)
    pct = n_out / len(df_clean) * 100
    outlier_summary[col] = (low, up, n_out)
    flag = "⚠️" if pct > 5 else ""
    print(f"{col:<28} {low:>8.2f} {up:>8.2f} {n_out:>10d} {pct:>7.1f}% {flag}")


In [ ]:
# --- 6.2 Boxplot trước & sau xử lý outlier ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Boxplot trước
df_clean[feature_cols_no_outcome].boxplot(ax=axes[0])
axes[0].set_title('Boxplot Trước Khi Xử Lý Outlier', fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)

# --- 6.3 Winsorization (clip outlier về ngưỡng IQR) ---
df_processed = df_clean.copy()
for col in feature_cols_no_outcome:
    low, up, _ = outlier_summary[col]
    df_processed[col] = df_processed[col].clip(lower=low, upper=up)

# Boxplot sau
df_processed[feature_cols_no_outcome].boxplot(ax=axes[1])
axes[1].set_title('Boxplot Sau Khi Winsorize Outlier', fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('outlier_treatment.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Outlier đã được xử lý bằng Winsorization (clip về ngưỡng IQR)")


## 7. Feature Engineering (Đề Xuất Bổ Sung)

In [ ]:
# --- 7.1 Tạo các đặc trưng mới có ý nghĩa y học ---
df_fe = df_processed.copy()

# BMI Category (WHO classification)
df_fe['BMI_Category'] = pd.cut(df_fe['BMI'], 
    bins=[0, 18.5, 24.9, 29.9, np.inf], 
    labels=[0, 1, 2, 3])  # 0=gầy, 1=bình thường, 2=thừa cân, 3=béo phì
df_fe['BMI_Category'] = df_fe['BMI_Category'].astype(int)

# Glucose Category (ADA guidelines)
df_fe['Glucose_Category'] = pd.cut(df_fe['Glucose'],
    bins=[0, 99, 125, np.inf],
    labels=[0, 1, 2])  # 0=bình thường, 1=tiền tiểu đường, 2=tiểu đường
df_fe['Glucose_Category'] = df_fe['Glucose_Category'].astype(int)

# Age Group
df_fe['Age_Group'] = pd.cut(df_fe['Age'],
    bins=[0, 30, 45, 60, np.inf],
    labels=[0, 1, 2, 3])  # 0=trẻ, 1=trung niên, 2=cao tuổi, 3=lớn tuổi
df_fe['Age_Group'] = df_fe['Age_Group'].astype(int)

# Insulin-Glucose Ratio (insulin resistance indicator)
df_fe['Insulin_Glucose_Ratio'] = df_fe['Insulin'] / (df_fe['Glucose'] + 1e-8)

# Pregnancies Risk (≥4 lần mang thai tăng nguy cơ)
df_fe['High_Pregnancy_Risk'] = (df_fe['Pregnancies'] >= 4).astype(int)

print("✅ Đã tạo các đặc trưng mới:")
new_features = ['BMI_Category', 'Glucose_Category', 'Age_Group', 'Insulin_Glucose_Ratio', 'High_Pregnancy_Risk']
for feat in new_features:
    print(f"   • {feat}")
print(f"\n📐 Shape sau Feature Engineering: {df_fe.shape}")


## 8. Feature Scaling (Chuẩn Hoá)

In [ ]:
# --- 8.1 StandardScaler (Z-score normalization) ---
# Không scale Outcome và các biến categorical mới
SCALE_COLS = [c for c in feature_cols_no_outcome] + ['Insulin_Glucose_Ratio']
CATEGORICAL_COLS = ['BMI_Category', 'Glucose_Category', 'Age_Group', 'High_Pregnancy_Risk', 'Outcome']

scaler = StandardScaler()
df_scaled = df_fe.copy()
df_scaled[SCALE_COLS] = scaler.fit_transform(df_fe[SCALE_COLS])

print("✅ StandardScaler áp dụng cho các features liên tục")
print("\n📊 Sau khi scale (Mean ≈ 0, Std ≈ 1):")
print(df_scaled[SCALE_COLS].describe().round(3).loc[['mean', 'std']].to_string())


## 9. Tổng Kết & So Sánh Trước/Sau Xử Lý

In [ ]:
# --- 9.1 So sánh phân phối trước/sau (original vs cleaned) ---
compare_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

fig, axes = plt.subplots(2, 5, figsize=(20, 7))

for i, col in enumerate(compare_cols):
    # Trước
    axes[0, i].hist(df[col], bins=25, color='#e74c3c', alpha=0.7, edgecolor='white')
    axes[0, i].set_title(f'{col}\n(Gốc)', fontweight='bold', fontsize=10)
    axes[0, i].set_ylabel('Tần suất' if i == 0 else '')
    
    # Sau
    axes[1, i].hist(df_processed[col], bins=25, color='#2ecc71', alpha=0.7, edgecolor='white')
    axes[1, i].set_title(f'{col}\n(Đã xử lý)', fontweight='bold', fontsize=10)
    axes[1, i].set_ylabel('Tần suất' if i == 0 else '')

axes[0, 0].set_ylabel('Trước — Tần suất', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Sau — Tần suất', fontsize=11, fontweight='bold')

plt.suptitle('So Sánh Phân Phối Trước & Sau Tiền Xử Lý', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('before_after_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- 9.2 Báo cáo tổng kết ---
print("=" * 60)
print("           📋 TỔNG KẾT TIỀN XỬ LÝ DỮ LIỆU")
print("=" * 60)
print(f"\n📥 Dataset gốc    : {df.shape[0]} mẫu × {df.shape[1]} cột")
print(f"📤 Dataset đã xử lý: {df_fe.shape[0]} mẫu × {df_fe.shape[1]} cột")
print(f"\n🔧 Các bước đã thực hiện:")
print("   1. ✅ Thay thế zero không hợp lệ → NaN (5 cột y tế)")
print("   2. ✅ Median imputation (Glucose, BloodPressure, BMI)")
print("   3. ✅ KNN Imputation k=5 (SkinThickness, Insulin)")
print("   4. ✅ Winsorization outlier (IQR × 1.5)")
print("   5. ✅ Feature Engineering (+5 đặc trưng mới)")
print("   6. ✅ StandardScaler (Z-score normalization)")
print(f"\n📊 Phân bố nhãn giữ nguyên:")
print(f"   Outcome=0: {(df_fe['Outcome']==0).sum()} ({(df_fe['Outcome']==0).mean()*100:.1f}%)")
print(f"   Outcome=1: {(df_fe['Outcome']==1).sum()} ({(df_fe['Outcome']==1).mean()*100:.1f}%)")
print("\n⚠️  Lưu ý cho XAI:")
print("   • Dùng df_processed (chưa scale) để visualize SHAP/LIME")
print("   • Dùng df_scaled (đã scale) để train model")
print("   • Class imbalance ~1.87:1 — cân nhắc class_weight='balanced'")


In [ ]:
# --- 9.3 Export dữ liệu đã xử lý ---
df_processed.to_csv('diabetes_cleaned.csv', index=False)
df_fe.to_csv('diabetes_engineered.csv', index=False)
df_scaled.to_csv('diabetes_scaled.csv', index=False)

print("✅ Đã lưu:")
print("   • diabetes_cleaned.csv     — Dữ liệu sau imputation & outlier treatment")
print("   • diabetes_engineered.csv  — Dữ liệu + feature engineering")
print("   • diabetes_scaled.csv      — Dữ liệu đã chuẩn hoá (ready for model)")


## 10. [Bonus] Gợi Ý Bước Tiếp Theo cho XAI

In [ ]:
# --- Gợi ý cài đặt thêm cho XAI ---
# !pip install shap lime xgboost

# Ví dụ quick preview với feature importance từ Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = df_scaled.drop('Outcome', axis=1)
y = df_scaled['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

print("📊 Model Performance (RandomForest - baseline):")
print(classification_report(y_test, rf.predict(X_test), 
      target_names=['Không TĐ', 'Có TĐ']))

# Feature Importance
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if v > importances.mean() else '#3498db' for v in importances.values]
importances.plot(kind='barh', ax=ax, color=colors, alpha=0.85)
ax.axvline(importances.mean(), color='gray', linestyle='--', alpha=0.8, label='Mean importance')
ax.set_title('Feature Importance — Random Forest\n(Baseline for XAI)', fontweight='bold', fontsize=13)
ax.set_xlabel('Importance Score')
ax.legend()
plt.tight_layout()
plt.savefig('feature_importance_rf.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 Bước tiếp theo: Áp dụng SHAP / LIME trên model này!")
